# Taobao User Behavior Data Cleaning & EDA

This notebook demonstrates the MVP workflow for cleaning Taobao-style user behavior data and calculating core e-commerce behavior metrics.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from data_cleaning import clean_user_behavior
from analysis import (
    behavior_type_distribution,
    cart_without_purchase_users,
    daily_active_users,
    daily_purchase_trend,
    high_value_users,
    hourly_activity,
    repurchase_analysis,
    summarize_behavior_metrics,
    top_categories_by_purchase,
)

sns.set_theme(style='whitegrid')

## 1. Clean Raw Data

In [ ]:
raw_path = PROJECT_ROOT / 'data' / 'raw_user_behavior_sample.csv'
cleaned_path = PROJECT_ROOT / 'data' / 'cleaned_user_behavior.csv'

df = clean_user_behavior(raw_path, cleaned_path)
df.head()

## 2. Core Metrics

In [ ]:
metrics = summarize_behavior_metrics(df)
pd.Series(metrics)

## 3. Behavior Distribution

In [ ]:
behavior_dist = behavior_type_distribution(df)
display(behavior_dist)

plt.figure(figsize=(7, 4))
sns.barplot(data=behavior_dist, x='behavior_type', y='behavior_count')
plt.title('Behavior Type Distribution')
plt.xlabel('Behavior Type')
plt.ylabel('Count')
plt.show()

## 4. Daily Active Users and Purchase Trend

In [ ]:
dau = daily_active_users(df)
purchase_trend = daily_purchase_trend(df)
display(dau)
display(purchase_trend)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.lineplot(data=dau, x='date', y='dau', marker='o', ax=axes[0])
axes[0].set_title('Daily Active Users')
axes[0].tick_params(axis='x', rotation=45)

sns.lineplot(data=purchase_trend, x='date', y='purchase_count', marker='o', ax=axes[1])
axes[1].set_title('Daily Purchase Trend')
axes[1].tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 5. Hourly User Activity

In [ ]:
hourly = hourly_activity(df)
hourly_pivot = hourly.pivot_table(index='behavior_type', columns='hour', values='behavior_count', fill_value=0)

plt.figure(figsize=(14, 4))
sns.heatmap(hourly_pivot, cmap='YlGnBu')
plt.title('Hourly Behavior Heatmap')
plt.xlabel('Hour')
plt.ylabel('Behavior Type')
plt.show()

## 6. Category, Repurchase, and User Value Analysis

In [ ]:
display(top_categories_by_purchase(df, top_n=10))
display(pd.Series(repurchase_analysis(df)))
display(cart_without_purchase_users(df).head(10))
display(high_value_users(df, top_n=10))

## 7. Funnel Summary

In [ ]:
funnel = pd.DataFrame({
    'step': ['pv_users', 'cart_users', 'buyer_users'],
    'users': [
        df.loc[df['behavior_type'] == 'pv', 'user_id'].nunique(),
        df.loc[df['behavior_type'] == 'cart', 'user_id'].nunique(),
        df.loc[df['behavior_type'] == 'buy', 'user_id'].nunique(),
    ],
})
funnel['conversion_from_pv'] = (funnel['users'] / funnel.loc[0, 'users']).round(4)
display(funnel)